#           **************Sales Analysis Project*****************

# Sales Performance Analysis

## Business Objective

### The objective of this project is to analyze sales data and identify top-performing products, cities, and business trends using Python and Pandas.

## Import libraries

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

## Load Dataset

In [19]:
df=pd.read_excel('Sales Analysis Data set.xlsx')
df

FileNotFoundError: [Errno 2] No such file or directory: 'Sales Analysis Data set.xlsx'

## Data Quality check

In [ ]:
# To understanding about rows and columns in dataset
print("Dataset Shape:", df.shape)
# To checking data types of all columns
display(df.dtypes)
# Checking any null values in all columns
display(df.isnull().sum())

#### There are No missing values were found in the dataset. Therefore, no missing-value treatment was required.

## Data Cleaning & Validation

In [ ]:
# Removing extra white spaces from columns name
df.columns=df.columns.str.strip()
df.columns

In [ ]:
# Convert Date column to datetime format
df["Date"] = pd.to_datetime(df["Date"])

# Duplicate rows check
print("Duplicate Rows:", df.duplicated().sum())

# Basic statistical summary of dataset
df.describe()

**Insights:** No exact duplicate records were found in the dataset. Therefore, no duplicate records were removed.

In [ ]:
# Calculate total sale as per given data.
df['Total_sale']=df['Qty']*df['Rate']
# Matching value of total sale
df['Total_match']=df['Sales_Amount']==df['Total_sale']
df['Total_match'].value_counts()

**Insights:** Sales amount was validated against Quantity × Rate. All 1,000 transaction records matched successfully.

In [ ]:
# Checking for negative rate/Qty
print("Qty <= 0:", (df['Qty'] <= 0).sum())
print("Rate <= 0:", (df['Rate'] <= 0).sum())

### Data Validation Summary

- No missing values were found.
- No exact duplicate records were found.
- No zero or negative quantity/rate values were found.
- Sales Amount matched Quantity × Rate for all 1,000 transactions.

The dataset is clean and ready for analysis.

## Feature Engineering
### Extract month from date column in 'YYYY-MM' format.
### Extract Year from date column


In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.to_period("M")
df[["Date", "Month"]].head()
df['Year']=df['Date'].dt.year
df

## Order_Size — Categorize on the basis of Qty
## Qty Order_Size 1–10 Small, 11–30 Medium, 31+ Large


In [ ]:
def qty_category(Qty):
    if Qty<=10:
        return 'SMALL'
    elif Qty<=30:
        return 'Medium'
    else:
        return 'LARGE'
df['Order_size']=df['Qty'].apply(qty_category)
df


## Total orders count in each category

In [ ]:
df_total_ordersize=df['Order_size'].value_counts().reset_index()
df_total_ordersize


## Overall sales performance
## Calculating Total Sales Revenue.

In [ ]:
df_totalsale=df['Total_sale'].sum()
print("Total Sales revenue is:" , df_totalsale)

## Calculating Total Quantity Sold.

In [ ]:
df_totalqty=df['Qty'].sum()
print("Total Sold Qty is:" , df_totalqty)

## Calculating Average Order Value.

In [ ]:
df_meanvalue=df['Total_sale'].mean()
print("Average order value is:" , df_meanvalue)

## Find out Highest and Lowest Single Order Value

In [ ]:
df_highest=df['Total_sale'].max()
df_lowest=df['Total_sale'].min()
print("Highest Single order value is:" , df_highest)
print("Lowest Single order value is:" , df_lowest)
df.loc[df['Sales_Amount']==df_lowest]
df.loc[df['Sales_Amount']==df_highest]

## Product-wise Sales Analysis

## Calculation perform for each product

### Total Sales
### Total Quantity Sold
### Average Sales per Order

In [ ]:
df_prod_wise_analysis=df.groupby(df['Product']).agg(
    Total_sale=('Total_sale','sum'),
    Total_qty_sold=('Qty','sum'),
    Average_sales_per_order=('Total_sale','mean')
).reset_index().round(2)
df_prod_wise_analysis

## 2. Ranking as per total sales.

In [ ]:
df_prod_wise_analysis['Sales_Rank']=df_prod_wise_analysis['Total_sale'].rank(ascending=False)
df_prod_wise_analysis.sort_values('Sales_Rank')

## 3. Top-performing product

In [ ]:
top_prod=df_prod_wise_analysis.loc[df_prod_wise_analysis['Sales_Rank']==1]
top_prod

## 4. Lowest revenue product

In [ ]:
low_rev_prod = df_prod_wise_analysis.sort_values('Total_sale',ascending=True).head(1)
low_rev_prod

## Top Product Contribution (%)
### Contribution %= Top Product sale/Total Sale*100

In [ ]:
df
cont_percentage=(top_prod['Total_sale'].iloc[0]/df_totalsale*100).round(2)
cont_percentage

In [ ]:
Top_prod_name=top_prod['Product'].iloc[0]
Top_prod_sale=top_prod['Total_sale'].iloc[0]
low_prod_name=low_rev_prod['Product'].iloc[0]
low_prod_sale=low_rev_prod['Total_sale'].iloc[0]
print(f'Top Performing Product is: {Top_prod_name} | Total Sale: {Top_prod_sale:,}')
print(f'Lowest Revenue Product is: {low_prod_name} | Total Sale: {low_prod_sale:,}')
print(f'Top Performing product {Top_prod_name} is contributed in Total Sale: {cont_percentage} %')

# City-wise Analysis
## City के आधार पर निकालिए:
### Total Sales
### Total Quantity Sold
### Total Orders
### Average Order Value

In [ ]:
df
city_wise_analysis=df.groupby(df['City']).agg(
    Total_Sale=('Total_sale','sum'),
    Total_qty_sold=('Qty','sum'),
    Average_order_value=('Total_sale','mean'),
    Total_Transaction=('Total_sale','count')
).reset_index().round(2)
city_wise_analysis



## Highest selling city

In [ ]:
highest_city = city_wise_analysis.loc[
    city_wise_analysis["Total_Sale"] == city_wise_analysis["Total_Sale"].max()
]
highest_city

## Lowes selling city

In [ ]:
lowest_city=city_wise_analysis.loc[city_wise_analysis["Total_Sale"] == city_wise_analysis['Total_Sale'].min()]
lowest_city

In [ ]:
highest_city_name=highest_city["City"].iloc[0]
lowest_city_name=lowest_city["City"].iloc[0]
highest_city_rev=highest_city["Total_Sale"].iloc[0]
lowest_city_rev=lowest_city["Total_Sale"].iloc[0]
highest_city_name,lowest_city_name,highest_city_rev,lowest_city_rev
print(f'Highest Revenue City: {highest_city_name} | Revenue: {highest_city_rev:,}')
print(f'Lowest Revenue City: {lowest_city_name} | Revenue: {lowest_city_rev:,}')

## Contribution % of Varanasi in Total Sales.
## Contribution % of Deoria in Total Sales.

In [ ]:
highest_contribution=(highest_city_rev/df_totalsale*100).round(2)
lowest_contribution=(lowest_city_rev/df_totalsale*100).round(2)
highest_contribution, lowest_contribution
print(f'{highest_city_name} contributed {highest_contribution} % of Total Sales.')
print(f'{lowest_city_name} contributes {lowest_contribution} % of Total Sales.')

## Sales ranking in City wise analysis

In [ ]:
city_wise_analysis['Sales_Rank']=city_wise_analysis['Total_Sale'].rank(ascending=False)
city_wise_analysis.sort_values('Sales_Rank')

## Marking revenue Category

In [ ]:
def rev_category(Sales_Rank):
    if Sales_Rank<=2.0:
        return 'High Revenue'
    elif Sales_Rank>2.0 and Sales_Rank<=4.0:
        return "Medium Revenue"
    else:
        return "Low Revenue"

city_wise_analysis['Revenue_Category']=city_wise_analysis['Sales_Rank'].apply(rev_category)
city_wise_analysis.sort_values('Sales_Rank')

## Find out Highest Ordered City and lowest ordered city with value and business insights

In [ ]:
city_wise_analysis
high_order=city_wise_analysis['Total_Transaction'].max()
low_order=city_wise_analysis['Total_Transaction'].min()
high_order,low_order
high_order_city=city_wise_analysis.loc[city_wise_analysis['Total_Transaction']==city_wise_analysis['Total_Transaction'].max()]
low_order_city=city_wise_analysis.loc[city_wise_analysis['Total_Transaction']==city_wise_analysis['Total_Transaction'].min()]
high_order_city
high_order_city_name=high_order_city.iloc[0]['City']
low_order_city_name=low_order_city.iloc[0]['City']
high_order_city_name,low_order_city_name

## Contribution % of each city in total sales

In [ ]:
city_wise_analysis['Revenue_Share_%'] = round((city_wise_analysis['Total_Sale'] / df_totalsale) * 100,2)
city_wise_analysis.sort_values('Sales_Rank')

## Business Insights

In [ ]:
print(f'Highest Ordered City: {high_order_city_name} | Total Order: {high_order}.')
print(f'Lowest Ordered City: {low_order_city_name} | Total Order: {low_order}.')
print()
print(f'{highest_city_name} generated the highest Revenue of Rs.{highest_city_rev:,}.')
print(f'{highest_city_name} Contributed {highest_contribution:,} % of Total Sales.')
print(f'{lowest_city_name} generated the lowest Revenue of Rs.{lowest_city_rev:,}.' )
print(f'{lowest_city_name} required focused marketing and sales improvement strategies.')

## Category wise analysis*************
### Find out category wise Total Sales,Total Quantity Sold,Average Order Value & Total Orders.

In [ ]:
df
category_wise_analysis=df.groupby(df['Category']).agg(
    Total_Sale=('Total_sale', 'sum'),
    Total_Qty_sold=('Qty', 'sum'),
    Avg_Order_value=('Total_sale','mean'),
    Total_Transaction=('Total_sale','count')
).reset_index().round(2)
category_wise_analysis

## Now Find out Highest and lowest revenue category

In [ ]:
category_wise_analysis
highest_rev_category=category_wise_analysis.loc[category_wise_analysis['Total_Sale']==category_wise_analysis['Total_Sale'].max()]
highest_rev_category
highest_category_name=highest_rev_category.iloc[0]['Category']
highest_category_rev=highest_rev_category.iloc[0]['Total_Sale']
highest_category_name, highest_category_rev
lowest_rev_category=category_wise_analysis.loc[category_wise_analysis['Total_Sale']==category_wise_analysis['Total_Sale'].min()]
lowest_rev_category
lowest_category_name=lowest_rev_category.iloc[0]['Category']
lowest_category_rev=lowest_rev_category.iloc[0]['Total_Sale']
lowest_category_name, lowest_category_rev
print(f'Highest Revenue Category: {highest_category_name} | Revenue: ₹{highest_category_rev:,}.')
print(f'Lowest Revenue Category: {lowest_category_name} | Revenue: ₹{lowest_category_rev:,}.')

## Now find out highest and lowest order category

In [ ]:
category_wise_analysis
highest_order_row=category_wise_analysis.loc[category_wise_analysis['Total_Transaction']==category_wise_analysis['Total_Transaction'].max()]
highest_order_row
lowest_order_row=category_wise_analysis.loc[category_wise_analysis['Total_Transaction']==category_wise_analysis['Total_Transaction'].min()]
lowest_order_row
highest_order_categ_name=highest_order_row.iloc[0]['Category']
lowest_order_categ_name=lowest_order_row.iloc[0]['Category']
highest_order_count=highest_order_row.iloc[0]['Total_Transaction']
lowest_order_count=lowest_order_row.iloc[0]['Total_Transaction']
print(f'Highest Order Category: {highest_order_categ_name} | Total Transaction: {highest_order_count:,}.')
print(f'Lowest Order Category: {lowest_order_categ_name} | Total Transaction: {lowest_order_count:,}.')

## Find out contribution % category wise

In [ ]:
category_wise_analysis
highest_category_contribution=(highest_category_rev/df_totalsale*100).round(2)
lowest_category_contribution=(lowest_category_rev/df_totalsale*100).round(2)
print(f'{highest_category_name} contributed {highest_category_contribution:,} % of Total Sales.')
print(f'{lowest_category_name} contributed {lowest_category_contribution:,} % of Total Sales.')


## Sales Ranking category wise

In [ ]:
category_wise_analysis['Sales_rank']=category_wise_analysis['Total_Sale'].rank(ascending=False)
category_wise_analysis

## Revenue category assign accourding to sales rank

In [ ]:
category_wise_analysis
def rev_category(Sales_rank):
    if Sales_rank==1.0:
        return "High Revenue"
    elif Sales_rank==2.0:
        return "Medium Revenue"
    else:
        return "Low Revenue"
category_wise_analysis["Rev_category"]=category_wise_analysis['Sales_rank'].apply(rev_category)
category_wise_analysis

## Revenue share calculate for every category

In [ ]:
category_wise_analysis
def calc_rev_share(Total_Sale):
    return (Total_Sale/df_totalsale*100).round(2)
category_wise_analysis['Rev_share %']=category_wise_analysis["Total_Sale"].apply(calc_rev_share)
category_wise_analysis

## Final Category wise Business Insights:----

In [ ]:
print(f'Highest Revenue Category: {highest_category_name} | Total Revenue: Rs.{highest_category_rev:,}.')
print(f'Lowest Revenue Category: {lowest_category_name} | Total Revenue: Rs.{lowest_category_rev:,}.')
print(f'Highest Ordered category is {highest_order_categ_name} | Total Transaction: {highest_order_count:,}.')
print(f'Lowest Ordered category is {lowest_order_categ_name} | Total Transaction: {lowest_order_count:,}.')
print()
print(f"**Conclusion**: {highest_category_name} category's products contribute {highest_category_contribution:,} % of Total Sales and all others category's products need to be focused on marketing strategy and also track the market behaviour/demand because {lowest_category_name} category's product contributed {lowest_category_contribution:,} % of Total Sales. So basically we have to need more focused on {lowest_category_name} category. ")

## Monthly Based Analysis*********
### Month
### Total Sales
### Total Quantity Sold
### Average Order Value
### Total Orders

In [ ]:
df['Date']=pd.to_datetime(df['Date'],format='%Y-%m-%d')
df['Month'] = df['Date'].dt.to_period('M')
df['Year']=df['Date'].dt.to_period('Y')
df
monthly_based_analysis=df.groupby(['Year','Month']).agg(
    Total_sale=('Total_sale','sum'),
    Total_qty_sold=('Qty','sum'),
    Avg_order_value=('Total_sale','mean'),
    Total_Transaction=('Total_sale','count')
).reset_index()
monthly_based_analysis['Month']=monthly_based_analysis['Month'].dt.strftime('%b-%y')
monthly_based_analysis['Avg_order_value']=monthly_based_analysis['Avg_order_value'].round(2)
monthly_based_analysis

## Finding out highest and lowest 'revenue month','Order Month' and revenue contribution percentage.

In [ ]:
monthly_based_analysis
highest_rev_month=monthly_based_analysis.loc[monthly_based_analysis['Total_sale']==monthly_based_analysis['Total_sale'].max()]
highest_monthly_revenue=highest_rev_month.iloc[0]['Total_sale']
lowest_rev_month=monthly_based_analysis.loc[monthly_based_analysis['Total_sale']==monthly_based_analysis['Total_sale'].min()]
lowest_monthly_revenue=lowest_rev_month.iloc[0]['Total_sale']
highest_order_month=monthly_based_analysis.loc[monthly_based_analysis['Total_Transaction']==monthly_based_analysis['Total_Transaction'].max()]
highest_monthly_order_name=highest_order_month.iloc[0]['Month']
lowest_order_month=monthly_based_analysis.loc[monthly_based_analysis['Total_Transaction']==monthly_based_analysis['Total_Transaction'].min()]
lowest_monthly_order_name=lowest_order_month.iloc[0]['Month']
highest_rev_month_name=highest_rev_month.iloc[0]['Month']
lowest_rev_month_name=lowest_rev_month.iloc[0]['Month']
highest_monthly_order_count=highest_order_month.iloc[0]['Total_Transaction']
lowest_monthly_order_count=lowest_order_month.iloc[0]['Total_Transaction']
highest_contribution_monthly=(highest_monthly_revenue/df_totalsale*100).round(2)
lowest_contribution_monthly=(lowest_monthly_revenue/df_totalsale*100).round(2)
print(f'Best Sales Month: {highest_rev_month_name} | Revenue: ₹ {highest_monthly_revenue:,}.')
print(f'Worst Sales Month: {lowest_rev_month_name} | Revenue: ₹ {lowest_monthly_revenue:,}.')
print(f'Highest order receive in Month: {highest_monthly_order_name} | Total Transaction: {highest_monthly_order_count:,}.')
print(f'Lowest order receive in Month: {lowest_monthly_order_name} | Total Transaction: {lowest_monthly_order_count:,}.')
print(f'Best Month {highest_rev_month_name} contributed {highest_contribution_monthly} % in Total Sale.')
print(f'Worst Month {lowest_rev_month_name} contributed {lowest_contribution_monthly} % in Total Sale.')


## Business Insights:

In [ ]:
print()

print(f"{highest_rev_month_name} recorded the highest monthly revenue of ₹{highest_monthly_revenue:,}, contributing {highest_contribution_monthly}% of the total sales.")

print(f"{lowest_rev_month_name} recorded the lowest monthly revenue of ₹{lowest_monthly_revenue:,}, contributing only {lowest_contribution_monthly}% of the total sales.")

print(f"{highest_monthly_order_name} received the highest number of customer orders {highest_monthly_order_count}, indicating strong customer demand.")

print(f"{lowest_monthly_order_name} received the fewest orders ({lowest_monthly_order_count}), suggesting that additional marketing or promotional efforts may be required during this period.")

# City wise revenue trend Visualisation

In [ ]:
city_chart = city_wise_analysis.sort_values('Total_Sale', ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(city_chart['City'],city_chart['Total_Sale'] / 100000,color='steelblue')
plt.title('City-wise Sales Revenue')
plt.xlabel('City')
plt.ylabel('Total Sales (₹ Lakhs)')
plt.grid(axis='y', alpha=0.2)

for i, value in enumerate(city_chart['Total_Sale'] / 100000):
    plt.text(i, value + 0.1, f'{value:.1f}', ha='center')

plt.tight_layout()
plt.savefig(r"C:\Users\admin\OneDrive\Desktop\sales-performance-analysis-python\Images\City_wise_revenue.png",dpi=300,bbox_inches='tight')
plt.show()

**Insight:** Varanasi generated the highest sales revenue of approximately ₹46.2 lakh, while Deoria generated the lowest revenue of approximately ₹36.7 lakh. The gap indicates an opportunity to investigate demand, product mix, and sales coverage in Deoria.

## Monthly trend Visualisation


### Monthly Sales Trend (Line Chart)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(monthly_based_analysis['Month'],monthly_based_analysis['Total_sale']/100000,marker='o',color='red')
plt.title('Monthly Sales Trend Analysis')
plt.xlabel('Month')
plt.ylabel('Total Sales in Lakh')
plt.grid(alpha=0.5)
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

## Month-on-Month (MoM) Sales Growth %

In [ ]:
monthly_based_analysis['MoM_Growth_%'] = (monthly_based_analysis['Total_sale'].pct_change() * 100).round(2)
monthly_based_analysis

## MoM Growth Chart

In [ ]:
plt.figure(figsize=(10,5))

colors = ['green' if value >= 0 else 'red'
          for value in monthly_based_analysis['MoM_Growth_%']]

plt.bar(monthly_based_analysis['Month'],monthly_based_analysis['MoM_Growth_%'],color=colors)
plt.title('Month-on-Month Sales Growth')
plt.xlabel('Month')
plt.ylabel('Growth (%)')
plt.xticks(rotation=60)
plt.grid(axis='y', alpha=0.2)
plt.show()

**Insight:** Sales growth was highly volatile. The strongest growth was recorded in November 2025 (+50.39%), while May 2026 showed the sharpest decline (-70.09%). This decline should be investigated for possible demand, inventory, pricing, or seasonal factors.

## Monthly Orders Bar Chart

In [ ]:
plt.figure(figsize=(10,5))
colors=[]
for order in monthly_based_analysis['Total_orders']:
    if order >60:
        colors.append('green')
    else:
        colors.append('red')
plt.bar(monthly_based_analysis['Month'],monthly_based_analysis['Total_orders'],color=colors)
plt.title('Monthly Order Analysis')
plt.xlabel('Months')
plt.ylabel('Total Orders')
plt.xticks(rotation=60)
plt.grid(axis='y',alpha=0.2)
plt.tight_layout()
for i, value in enumerate(monthly_based_analysis['Total_orders']):
    plt.text(i, value + 1, str(value), ha='center', fontsize=9)
plt.show()

## Monthly Quantity Sold (Bar Chart)


In [ ]:
plt.figure(figsize=(10,5))
colors=[]
for Quantity in monthly_based_analysis['Total_qty_sold']:
    if Quantity >1500:
        colors.append('green')
    elif Quantity<1500 and Quantity >1000:
        colors.append('skyblue')
    else:
        colors.append('red')
plt.bar(monthly_based_analysis['Month'],monthly_based_analysis['Total_qty_sold'],color=colors)
plt.title('Monthly Quantity Sold Analysis')
plt.xlabel('Months')
plt.ylabel('Quantity Sold')
plt.xticks(rotation=60)
plt.yticks(np.arange(100,2000,250))
plt.grid(axis='y',alpha=0.2)
plt.tight_layout()
for i, value in enumerate(monthly_based_analysis['Total_qty_sold']):
    plt.text(i, value + 30, str(value), ha='center', fontsize=9)
plt.show()

## Average Order Value Trend (Line Ch)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(monthly_based_analysis['Month'],monthly_based_analysis['Avg_order_value'],marker='o',color='red')
plt.title('Month wise Average Order Value Trend Analysis')
plt.grid(alpha=0.5)
plt.xlabel('Month')
plt.xticks(rotation=60)
plt.ylabel('Average order value')
plt.tight_layout()
plt.show()

## Category Revenue Share (Pie Chart)

In [ ]:
plt.figure(figsize=(5,5))
plt.pie(category_wise_analysis['Rev_share %'],labels=category_wise_analysis['Category'],autopct='%.2f%%',startangle=90,explode=[0.0,0.0,0.1],shadow=True)
plt.title('Category wise Revenue Share Percentage')
plt.axis('equal')
plt.show()


## Product-wise Revenue Analysis

In [ ]:
product_chart = df_prod_wise_analysis.sort_values('Total_sale',ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(product_chart['Product'],product_chart['Total_sale'] / 100000,color='darkorange')
plt.title('Product-wise Sales Revenue')
plt.xlabel('Product')
plt.ylabel('Total Sales (₹ Lakhs)')
plt.grid(axis='y', alpha=0.2)

for i, value in enumerate(product_chart['Total_sale'] / 100000):
    plt.text(i, value + 0.1, f'{value:.1f}', ha='center')

plt.tight_layout()
plt.show()

**Insight:** Pesticide was the highest-revenue product, generating approximately ₹46.7 lakh in sales. Urea was the lowest-revenue product at approximately ₹35.5 lakh. However, the difference between products is relatively small, indicating a balanced product-revenue mix.

# Final Business Summary

## Key Business Findings

- The analysis covered 1,000 clean sales transactions from January 2025 to May 2026, with no missing values, duplicate records, invalid quantity/rate values, or sales-calculation mismatches.
- Total sales revenue was approximately ₹2.03 crore across five cities and five agricultural products.
- Pesticide was the top-performing product, generating ₹46.72 lakh and contributing 23% of total sales. Urea generated the lowest product revenue at ₹35.54 lakh.
- Varanasi was the highest-revenue city at approximately ₹46.2 lakh, while Deoria recorded the lowest revenue at approximately ₹36.7 lakh.
- Sales performance showed month-to-month volatility. November 2025 recorded the strongest monthly growth (+50.39%), while May 2026 showed the largest decline (-70.09%).
- The business should investigate the lower performance in Deoria and the sharp decline in May 2026 by reviewing product demand, inventory availability, pricing, and whether the May 2026 data represents a complete month.

## Business Recommendations

1. Analyze the product mix and sales approach used in Varanasi and identify practices that can be applied in Deoria.
2. Review Urea sales by city and month to identify whether lower revenue is caused by lower demand, pricing, or product availability.
3. Investigate the sales decline in May 2026 before taking action, including data-period completeness, stock availability, and seasonal demand.
4. Track Month-on-Month growth regularly to detect sales declines early and support timely business decisions.